# GEDI Footprint Extraction Workflow (1 of 2 — Data Extraction)

Extracts GEDI Level 2A shots over an area of interest, filters to high-quality returns, and exports point and footprint (25 m buffer) GeoJSON files.

**Bounding box:** Set from a `.gdb` boundary file (interactive dialog) **or** entered manually below.

**This is notebook 1 of 2.** It covers Steps 1-9: everything needed to pull GEDI data, filter it, and clip it to your study area. Once this finishes, open **`gedi_footprint_2_analysis.ipynb`** for the Monte Carlo simulation, CHM comparisons, and plots — that notebook loads the files saved here, so you never need to re-run the GEDI search just to keep analyzing the data.

---
## Setup
Ensure kernel is set to Python 3  
Install required packages if you haven't already:

```bash
# in Google Colab:
#   !pip install earthaccess h5py numpy pandas geopandas shapely fiona -q
# in VS Code terminal:
#   python -m pip install earthaccess h5py numpy pandas geopandas shapely fiona
```

In [7]:
#------------ imports ----------------------------------------------------------
print("Importing libraries...")
import earthaccess
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import datetime
import os
import tkinter as tk
from tkinter import filedialog, simpledialog
import fiona
from pyproj import CRS, Transformer



Importing libraries...


---
## Step 1: Set Output Folder and File Header

A dialog will open asking you to:
1. Select the folder where output files (CSV, GeoJSON) will be saved
2. Enter a short header used to name all output files (e.g. `bpines` → `bpines_gedi_shots.csv`)

In [8]:
data_folder = input("Enter output folder path: ").strip().strip('"')
file_header = input("Enter file name header (e.g. 'bpines'): ").strip().lower().replace(" ", "_")

csv_path            = os.path.join(data_folder, f'{file_header}_gedi_shots.csv')
geojson_path        = os.path.join(data_folder, f'{file_header}_gedi_shots.geojson')
buffer_geojson_path = os.path.join(data_folder, f'{file_header}_gedi_footprints.geojson')

print(f"\nOutput files will be:")
print(f"  {csv_path}")
print(f"  {geojson_path}")
print(f"  {buffer_geojson_path}")


Output files will be:
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.csv
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.geojson
  C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_footprints.geojson


---
## Step 2: Define Area of Interest Bounding Box

**Option A** — Load from a `.gdb` boundary file (recommended) or select a GeoJSON file: run the cell below.

**Option B** — Enter coordinates manually: skip to the manual cell further below.

### Option A: Load boundary from .gdb or GeoJSON

In [12]:
# Select the boundary file (.gdb folder OR a single-layer file like .geojson/.shp)
boundary_path = input("Paste the path to your boundary file (.gdb folder, .geojson, or .shp): ").strip().strip('"')
if not boundary_path:
    raise SystemExit("No boundary path entered. Exiting.")

if boundary_path.lower().endswith('.gdb'):
    # Geodatabase — may contain multiple layers, ask which one to use
    layers = fiona.listlayers(boundary_path)
    print("\nLayers found in geodatabase:")
    for i, layer in enumerate(layers):
        print(f"  {i + 1}: {layer}")

    layer_index = int(input(f"\nEnter the number of the layer to use as the clipping boundary (1-{len(layers)}): ").strip())
    if not layer_index:
        raise SystemExit("No layer selected. Exiting.")
    layer_name = layers[layer_index - 1]
    print(f"\nUsing layer: {layer_name}")

    boundary = gpd.read_file(boundary_path, layer=layer_name)
else:
    # Single-layer file (.geojson, .shp, etc.) — no layer selection needed
    boundary = gpd.read_file(boundary_path)
    print(f"\nUsing boundary file: {boundary_path}")

# Load the boundary layer and compute bounding box
boundary_wgs84 = boundary.to_crs("EPSG:4326")
minx, miny, maxx, maxy = boundary_wgs84.total_bounds
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = minx, miny, maxx, maxy

print(f"\nBounding box derived from boundary:")
print(f"LAT_MIN, LAT_MAX = {LAT_MIN:.6f}, {LAT_MAX:.6f}")
print(f"LON_MIN, LON_MAX = {LON_MIN:.6f}, {LON_MAX:.6f}")


Using boundary file: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\GitHub_Repository\Biomass_Monitoring_Research\bartleson_boundary.geojson

Bounding box derived from boundary:
LAT_MIN, LAT_MAX = 35.079525, 35.096023
LON_MIN, LON_MAX = -120.553011, -120.530919


### Option B: Insert coordinates manually

Only run this cell if you skipped Option A.

In [4]:
# # -------- Insert coordinates for area of interest -----------------------------
# LAT_MIN, LAT_MAX = 35.309, 35.312
# LON_MIN, LON_MAX = -120.664, -120.657

# # Create a boundary polygon from the manual coordinates (needed for clipping in Step 9)
# from shapely.geometry import box
# boundary = gpd.GeoDataFrame(geometry=[box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)], crs="EPSG:4326")

---
## Step 3: Login to NASA Earthdata

In [13]:
# ---------- Login to NASA Earthdata -------------------------------------------
# prompts for NASA Earthdata username and password
earthaccess.login()

---
## Step 4: Search for GEDI L2A (Height) Granules

In [ ]:
# ---------- Search for GEDI granules over area of interest --------------------
print('Searching for GEDI granules...')
results = earthaccess.search_data(
    short_name='GEDI02_A',
    version='002',
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
)
print(f'Found {len(results)} granules covering your Area of Interest.')

Searching for GEDI granules...
Found 28 granules covering your arboretum


c:\Users\davisk10\miniconda3\Lib\site-packages\earthaccess\results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


---
## Step 4b: Search for GEDI L4A (Biomass) Granules

GEDI L4A provides footprint-level **aboveground biomass density (AGBD)** predictions generated from the same L2A waveforms — each L4A shot carries the identical `shot_number` as its L2A counterpart, which is what lets the two products be joined exactly (no spatial matching needed) in Step 6b. Searching the same bounding box as Step 4.

In [15]:
# ---------- Search for GEDI L4A (biomass) granules over area of interest ------
# NOTE: L4A is hosted by ORNL DAAC (not LP DAAC like L2A), and its CMR collection
# version is '2.1' -- NOT '002' like GEDI02_A. Using '002' here returns 0 granules.
print('Searching for GEDI L4A (biomass) granules...')
results_biomass = earthaccess.search_data(
    doi='10.3334/ORNLDAAC/2056',
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
)
print(f'Found {len(results_biomass)} L4A granules covering your area of interest')

# Fallback: if this still finds 0 granules (e.g. version string changes again),
# drop the version constraint and let CMR return whatever is current.
if len(results_biomass) == 0:
    print('No granules with version 2.1 -- retrying without a version constraint...')
    results_biomass = earthaccess.search_data(
        short_name='GEDI04_A',
        bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
    )
    print(f'Found {len(results_biomass)} L4A granules covering your area of interest')

Searching for GEDI L4A (biomass) granules...


c:\Users\davisk10\miniconda3\Lib\site-packages\earthaccess\search.py:946: FutureWarning: As of version 1.0, `DataCollection.concept_id` will be accessed as an attribute; e.g. use `DataCollection.concept_id` **not** `DataCollection.concept_id()`
  concept_id = collection[0].concept_id()


Found 28 L4A granules covering your area of interest


---
## Step 5: Stream Granules and Extract Shots

Streams each granule directly — no download needed. (~20 min depending on granule count)

In [16]:
# --- Stream each granule and extract shots (~20 min) --------------------------
all_shots = []
waveform_store = {}   # stores raw waveform arrays keyed by shot_number
                      # kept separate from the DataFrame because each shot has
                      # a variable number of waveform samples (~200–2000 values)

for granule in results:
    filename = granule['meta']['native-id']
    print(f'\nProcessing: {filename}')

    try:
        # Stream the file directly — no download needed
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                # Filter to Area of Interest bounding box
                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                print(f'  {beam}: {mask.sum()} shots in Area of Interest')

                rh = h5file[beam]['rh'][mask]

                # Date from filename
                try:
                    year = int(filename[9:13])
                    doy  = int(filename[13:16])
                    date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
                    date_str = date.strftime('%Y-%m-%d')
                except:
                    date_str = 'unknown'

                shot_data = {
                    'filename':     filename,
                    'beam':         beam,
                    'date':         date_str,
                    'lat':          lat[mask],
                    'lon':          lon[mask],
                    'rh25':         rh[:, 25],
                    'rh50':         rh[:, 50],
                    'rh75':         rh[:, 75],
                    'rh85':         rh[:, 85],
                    'rh95':         rh[:, 95],  
                    'rh96':         rh[:, 96],
                    'rh97':         rh[:, 97],
                    'rh98':         rh[:, 98], #relative height at 98th percentile (top of forest canopy height in m)
                    'rh99':         rh[:, 99],
                    'rh100':        rh[:, 100],
                    'sensitivity':  h5file[beam]['sensitivity'][mask], #how well detected ground return through the canopy, ranges from 0 to 1
                                                                        #0.9 - 1.0 → excellent — laser punched through canopy cleanly and found ground
                    'quality_flag': h5file[beam]['quality_flag'][mask],
                    'degrade_flag': h5file[beam]['degrade_flag'][mask],
                    'shot_number':  h5file[beam]['shot_number'][mask].astype(str),
                }

                all_shots.append(pd.DataFrame(shot_data))
                # ── Extract raw waveforms for shots in bounding box ───────────
                # rxwaveform is one long 1D array for the entire beam —
                # rx_sample_start_index and rx_sample_count tell us where
                # each individual shot's waveform starts and how long it is
                rx_waveform   = h5file[beam]['rxwaveform'][:]
                start_indices = h5file[beam]['rx_sample_start_index'][mask]
                sample_counts = h5file[beam]['rx_sample_count'][mask]
                shot_numbers  = h5file[beam]['shot_number'][mask]   # ← add this line

                for i in range(mask.sum()):
                    sn    = str(shot_numbers[i])
                    start = start_indices[i]
                    count = sample_counts[i]
                    # slice out just this shot's waveform from the full beam array
                    waveform_store[sn] = rx_waveform[start : start + count]


            except KeyError as e:
                print(f'  Skipping {beam} — missing field: {e}')
                continue

        h5file.close()

    except Exception as e:
        print(f'  Error: {e}')
        continue


Processing: GEDI02_A_2019263093017_O04371_03_T04964_02_003_01_V002


c:\Users\davisk10\miniconda3\Lib\site-packages\earthaccess\store.py:523: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum([granule.size() for granule in granules]) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]


Processing: GEDI02_A_2025177154916_O37032_03_T03541_02_004_02_V002


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

---
## Step 5b: Stream L4A Granules and Extract Biomass

Same streaming approach as Step 5, but reading GEDI04_A biomass fields instead of L2A height metrics. Key fields:

- **`agbd`** — predicted aboveground biomass density (Mg/ha)
- **`agbd_se`** — standard error of the AGBD prediction (Mg/ha)
- **`l4_quality_flag`** — 1 = shot is a reliable sample of the population the biomass model was trained on (this is separate from, and in addition to, the L2A `quality_flag` — filter on this downstream if you want the high-confidence biomass subset)
- **`algorithm_run_flag`** — 1 = the L4A algorithm actually produced a prediction for this shot (0 = no estimate, e.g. over water)

Because `shot_number` matches its L2A counterpart exactly, this is what lets us merge the two datasets in Step 6b.

In [17]:
# --- Stream each L4A granule and extract biomass shots ------------------------
all_biomass = []

for granule in results_biomass:
    filename = granule['meta']['native-id']
    print(f'\nProcessing: {filename}')

    try:
        # Stream the file directly — no download needed
        files = earthaccess.open([granule])
        h5file = h5py.File(files[0], 'r')
        beams = [k for k in h5file.keys() if k.startswith('BEAM')]

        for beam in beams:
            try:
                lat = h5file[beam]['lat_lowestmode'][:]
                lon = h5file[beam]['lon_lowestmode'][:]

                # Filter to the same bounding box used for L2A
                mask = (
                    (lat >= LAT_MIN) & (lat <= LAT_MAX) &
                    (lon >= LON_MIN) & (lon <= LON_MAX)
                )

                if mask.sum() == 0:
                    continue

                print(f'  {beam}: {mask.sum()} shots in area of interest')

                biomass_data = {
                    'shot_number':       h5file[beam]['shot_number'][mask].astype(str),
                    'agbd':              h5file[beam]['agbd'][mask],              # aboveground biomass density (Mg/ha)
                    'agbd_se':           h5file[beam]['agbd_se'][mask],           # standard error of agbd (Mg/ha)
                    'l4_quality_flag':   h5file[beam]['l4_quality_flag'][mask],   # 1 = reliable AGBD sample
                    'algorithm_run_flag':h5file[beam]['algorithm_run_flag'][mask],# 1 = L4A model produced a prediction
                    'sensitivity_l4a':   h5file[beam]['sensitivity'][mask],
                    'degrade_flag_l4a':  h5file[beam]['degrade_flag'][mask],
                }

                all_biomass.append(pd.DataFrame(biomass_data))

            except KeyError as e:
                print(f'  Skipping {beam} — missing field: {e}')
                continue

        h5file.close()

    except Exception as e:
        print(f'  Error: {e}')
        continue

if len(all_biomass) == 0:
    biomass_df = pd.DataFrame(columns=['shot_number', 'agbd', 'agbd_se', 'l4_quality_flag',
                                        'algorithm_run_flag', 'sensitivity_l4a', 'degrade_flag_l4a'])
    print('\nNo L4A biomass shots found in area of interest')
else:
    biomass_df = pd.concat(all_biomass, ignore_index=True)
    biomass_df['shot_number'] = biomass_df['shot_number'].astype(str)
    print(f'\nTotal L4A biomass shots found: {len(biomass_df)}')


Processing: GEDI_L4A_AGB_Density_V2_1.GEDI04_A_2019263093017_O04371_03_T04964_02_002_02_V002.h5


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

  BEAM0000: 29 shots in area of interest
  BEAM0001: 45 shots in area of interest
  BEAM0010: 30 shots in area of interest
  BEAM0011: 9 shots in area of interest

Processing: GEDI_L4A_AGB_Density_V2_1.GEDI04_A_2020018025532_O06228_02_T00109_02_002_02_V002.h5


...


Processing: GEDI_L4A_AGB_Density_V2_1.GEDI04_A_2025177154916_O37032_03_T03541_02_004_01_V002.h5


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]


Total L4A biomass shots found: 1216


---
## Step 6: Filter to High-Quality Shots and Save Outputs

In [18]:
# ------------- Combine all shots ----------------------------------------------
if len(all_shots) == 0:
    print('\nNo shots found in Area of Interest.')
else:
    combined = pd.concat(all_shots, ignore_index=True)
    print(f'\nTotal shots found: {len(combined)}')

    # Filter to high quality shots only
    quality = combined[
        (combined['quality_flag'] == 1) &
        (combined['degrade_flag'] == 0)
    ].copy()
    print(f'High quality shots: {len(quality)}')
    print(quality[['date', 'beam', 'lat', 'lon', 'rh98', 'sensitivity']].to_string())

    # Ensure shot_number stays as string (ArcGIS Pro compatibility)
    quality['shot_number'] = quality['shot_number'].astype(str)

    # Save CSV
    quality.to_csv(csv_path, index=False)
    print(f'\nCSV saved: {csv_path}')


Total shots found: 1216
High quality shots: 519
            date      beam        lat         lon       rh98  sensitivity
0     2019-09-20  BEAM0000  35.095913 -120.544120   3.400000     0.951063
1     2019-09-20  BEAM0000  35.095563 -120.543661   3.100000     0.938111
2     2019-09-20  BEAM0000  35.095212 -120.543202   3.250000     0.957686
3     2019-09-20  BEAM0000  35.094862 -120.542745   3.180000     0.957387
4     2019-09-20  BEAM0000  35.094511 -120.542286   3.250000     0.952875
5     2019-09-20  BEAM0000  35.094161 -120.541827   3.590000     0.952596
6     2019-09-20  BEAM0000  35.093809 -120.541367   3.970000     0.943250
7     2019-09-20  BEAM0000  35.093458 -120.540906   3.820000     0.939770
8     2019-09-20  BEAM0000  35.093108 -120.540449   4.790000     0.924245
9     2019-09-20  BEAM0000  35.092758 -120.539992   2.500000     0.935446
10    2019-09-20  BEAM0000  35.092406 -120.539532   3.630000     0.940222
11    2019-09-20  BEAM0000  35.092055 -120.539072   3.850000   

*Note: the extraction/filtering work ends at Step 9 below. All further analysis (Monte Carlo simulation, CHM comparisons, plots) lives in the companion notebook `gedi_footprint_2_analysis.ipynb`, which loads the files saved here instead of re-running the GEDI search.*

---
## Step 6b: Merge Biomass into the Quality Shot Table

Joins the L4A biomass fields onto `quality` (the L2A height table from Step 6) using `shot_number` — since it's the same lidar shot in both products, this is an exact key match rather than a spatial join.

This is a **left join**, so every L2A shot in `quality` is kept even where GEDI04_A produced no biomass estimate (e.g. `algorithm_run_flag = 0` over water or non-forested/unsupported cover types) — those rows simply get `NaN` in the biomass columns.

This step does **not** filter on `l4_quality_flag`. Apply that afterward if your analysis needs only the high-confidence biomass subset, e.g.:
```python
high_conf_biomass = quality[quality['l4_quality_flag'] == 1]
```

In [19]:
# ------------- Merge L4A biomass into the quality shot table ------------------
quality = quality.merge(biomass_df, on='shot_number', how='left')

matched = quality['agbd'].notna().sum()
print(f'L2A shots with a matching L4A biomass estimate: {matched} / {len(quality)}')
print(quality[['date', 'beam', 'lat', 'lon', 'rh98', 'agbd', 'agbd_se', 'l4_quality_flag']].to_string())

# Re-save the CSV now that biomass columns are included
quality.to_csv(csv_path, index=False)
print(f'\nCSV re-saved with biomass columns: {csv_path}')

L2A shots with a matching L4A biomass estimate: 519 / 519
           date      beam        lat         lon       rh98         agbd    agbd_se  l4_quality_flag
0    2019-09-20  BEAM0000  35.095913 -120.544120   3.400000     2.602650   2.999727                1
1    2019-09-20  BEAM0000  35.095563 -120.543661   3.100000     2.014405   3.002018                0
2    2019-09-20  BEAM0000  35.095212 -120.543202   3.250000     2.299228   3.000855                1
3    2019-09-20  BEAM0000  35.094862 -120.542745   3.180000     2.163994   3.001394                1
4    2019-09-20  BEAM0000  35.094511 -120.542286   3.250000     2.299228   3.000855                1
5    2019-09-20  BEAM0000  35.094161 -120.541827   3.590000     3.013623   2.998349                1
6    2019-09-20  BEAM0000  35.093809 -120.541367   3.970000     3.924572   2.995762                0
7    2019-09-20  BEAM0000  35.093458 -120.540906   3.820000     3.550838   2.996757                0
8    2019-09-20  BEAM0000  35.093

---
## Step 7: Export Center Coordinate Points of Footprints to GeoJSON

In [20]:
# ── Step 7: Build point GeoDataFrame ──────────────────────────────────────────
if len(quality) == 0:
    print('No quality shots to export. Exiting.')
else:
    geometry_pts = [Point(lon, lat) for lon, lat in zip(quality['lon'], quality['lat'])]
    gdf_points = gpd.GeoDataFrame(quality, geometry=geometry_pts, crs='EPSG:4326')

    # Save point GeoJSON
    gdf_points.to_file(geojson_path, driver='GeoJSON')
    print(f'Point GeoJSON saved: {geojson_path}')

Point GeoJSON saved: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.geojson


---
## Step 8: Create 25 m Footprint Buffers and Export

**WHY WE REPROJECT:**  
GEDI points are stored in WGS84 (EPSG:4326), where coordinates are in degrees. Buffering in degrees does NOT give you metres — 0.0001° of longitude is ~9 m near the equator but changes with latitude.

Solution: reproject to a CRS measured in metres, buffer by exactly 12.5 m (radius), then reproject back to WGS84.

UTM Zone 10N (EPSG:32610) covers central California and is appropriate for this study area. Adjust the EPSG code if your ROI is in a different UTM zone.

**WHAT THE BUFFER DOES:**  
Each GEDI shot records a single lat/lon centre point, but the real laser footprint illuminates a ~25 m diameter circle on the ground. `buffer(12.5)` expands each point into a circle with radius 12.5 m, giving a 25 m diameter polygon that matches the physical footprint. When you later extract biomass, canopy height, or image pixel values WITHIN these polygons, you are sampling the same area the lidar saw.

In [21]:
# ── Step 8: Create 25 m footprint buffers ─────────────────────────────────────
UTM_CRS = 'EPSG:32610'   # UTM Zone 10N — change if your site is elsewhere

gdf_utm = gdf_points.to_crs(UTM_CRS)
gdf_utm['geometry'] = gdf_utm.geometry.buffer(12.5)    # 12.5 m radius → 25 m diameter
gdf_footprints = gdf_utm.to_crs('EPSG:4326')           # reproject back to WGS84

gdf_footprints.to_file(buffer_geojson_path, driver='GeoJSON')
print(f'Footprint GeoJSON saved (25 m buffers): {buffer_geojson_path}')

print('\nDone! Load the GeoJSON files into QGIS or ArcGIS Pro.')
print('Outputs:')
print(f'  Points     → {geojson_path}')
print(f'  Footprints → {buffer_geojson_path}')
print(f'  CSV        → {csv_path}')

Footprint GeoJSON saved (25 m buffers): C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_footprints.geojson

Done! Load the GeoJSON files into QGIS or ArcGIS Pro.
Outputs:
  Points     → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.geojson
  Footprints → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_footprints.geojson
  CSV        → C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.csv


---
## Diagnostic: Why Were Shots Rejected?

Re-streams all granules without quality filtering and prints a breakdown of rejection reasons. Run this if you want to understand why shots were excluded.

In [ ]:
# # ------ Diagnostic: why were shots rejected? ----------------------------------

# all_shots_unfiltered = []

# for granule in results:
#     filename = granule['meta']['native-id']

#     try:
#         files = earthaccess.open([granule])
#         h5file = h5py.File(files[0], 'r')
#         beams = [k for k in h5file.keys() if k.startswith('BEAM')]

#         for beam in beams:
#             try:
#                 lat = h5file[beam]['lat_lowestmode'][:]
#                 lon = h5file[beam]['lon_lowestmode'][:]

#                 mask = (
#                     (lat >= LAT_MIN) & (lat <= LAT_MAX) &
#                     (lon >= LON_MIN) & (lon <= LON_MAX)
#                 )

#                 if mask.sum() == 0:
#                     continue

#                 quality_flag = h5file[beam]['quality_flag'][mask]
#                 degrade_flag = h5file[beam]['degrade_flag'][mask]
#                 sensitivity  = h5file[beam]['sensitivity'][mask]

#                 # Date from filename
#                 try:
#                     year = int(filename[9:13])
#                     doy  = int(filename[13:16])
#                     date = datetime.datetime(year, 1, 1) + datetime.timedelta(doy - 1)
#                     date_str = date.strftime('%Y-%m-%d')
#                 except:
#                     date_str = 'unknown'

#                 for i in range(mask.sum()):
#                     all_shots_unfiltered.append({
#                         'filename':     filename,
#                         'beam':         beam,
#                         'date':         date_str,
#                         'lat':          lat[mask][i],
#                         'lon':          lon[mask][i],
#                         'quality_flag': quality_flag[i],
#                         'degrade_flag': degrade_flag[i],
#                         'sensitivity':  sensitivity[i],
#                         'rejected':     quality_flag[i] != 1 or degrade_flag[i] != 0
#                     })

#             except KeyError as e:
#                 continue

#         h5file.close()

#     except Exception as e:
#         print(f'Error: {e}')
#         continue

# # ── Summary ───────────────────────────────────────────────────────────────────
# df_all = pd.DataFrame(all_shots_unfiltered)

# print(f'Total shots in bounding box: {len(df_all)}')
# print(f'Accepted (quality=1, degrade=0): {len(df_all[~df_all["rejected"]])}')
# print(f'Rejected: {len(df_all[df_all["rejected"]])}')
# print()
# print('Rejection breakdown:')
# print(f'  quality_flag = 0: {len(df_all[df_all["quality_flag"] == 0])}')
# print(f'  degrade_flag != 0: {len(df_all[df_all["degrade_flag"] != 0])}')
# print()
# print('Rejected shots by date and beam:')
# rejected = df_all[df_all['rejected']]
# print(rejected[['date', 'beam', 'lat', 'lon', 'quality_flag', 'degrade_flag', 'sensitivity']].to_string())

---
## Step 9: Clip GEDI Footprints to Previously Entered Study Area Bounday 

In [22]:
## Step 9: Clip Footprints to Study Area Boundary

# Reproject boundary to WGS84 to match footprints
boundary_wgs84 = boundary.to_crs("EPSG:4326")
boundary_union = boundary_wgs84.union_all()  # single geometry for comparison

# Keep only footprints completely within the boundary
gdf_footprints_clipped = gdf_footprints[gdf_footprints.geometry.within(boundary_union)].copy()

# Match points to surviving footprints using shot_number
gdf_points_clipped = gdf_points[gdf_points['shot_number'].isin(gdf_footprints_clipped['shot_number'])].copy()

print(f"Footprints before filtering:      {len(gdf_footprints)}")
print(f"Footprints fully within boundary: {len(gdf_footprints_clipped)}")
print(f"Points matched to footprints:     {len(gdf_points_clipped)}")

# Save outputs
clipped_footprints_path = os.path.join(data_folder, f'{file_header}_gedi_footprints_clipped.geojson')
clipped_points_path     = os.path.join(data_folder, f'{file_header}_gedi_shots_clipped.geojson')

gdf_footprints_clipped.to_file(clipped_footprints_path, driver='GeoJSON')
gdf_points_clipped.to_file(clipped_points_path, driver='GeoJSON')

print(f"\nClipped footprints saved: {clipped_footprints_path}")
print(f"Clipped points saved:     {clipped_points_path}")

Footprints before filtering:      519
Footprints fully within boundary: 225
Points matched to footprints:     225

Clipped footprints saved: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_footprints_clipped.geojson
Clipped points saved:     C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots_clipped.geojson


---
## Done — Outputs for the Analysis Notebook

This notebook's job ends here. The cell below saves one more file — a flat CSV version of the clipped points (handy for opening directly in Excel) — and prints out every file that was produced.

**To continue the analysis (Monte Carlo simulation, CHM comparisons, plots, etc.), open `gedi_footprint_2_analysis.ipynb`.** It does *not* need this notebook to still be running — its first code cell opens a file-picker dialog so you can select the outputs saved below, any time, in a fresh kernel.

In [23]:
# ── Save a flat CSV of the clipped points (no geometry) for quick viewing ─────
clipped_csv_path = os.path.join(data_folder, f'{file_header}_gedi_shots_clipped.csv')
gdf_points_clipped.drop(columns='geometry', errors='ignore').to_csv(clipped_csv_path, index=False)

print('All outputs saved to:', data_folder)
print(f'  All quality shots (unclipped, height + biomass) : {csv_path}')
print(f'  All quality points (GeoJSON)                    : {geojson_path}')
print(f'  All footprints (GeoJSON)                        : {buffer_geojson_path}')
print(f'  Clipped points (GeoJSON)                        : {clipped_points_path}')
print(f'  Clipped footprints (GeoJSON)                    : {clipped_footprints_path}')
print(f'  Clipped points (flat CSV, height + biomass)     : {clipped_csv_path}')
print()
print('agbd = aboveground biomass density (Mg/ha); rh98 = canopy height (m).')
print('Open gedi_footprint_2_analysis.ipynb to continue — it will ask you')
print('to select the clipped points GeoJSON file above, and will auto-find')
print('the matching clipped footprints file in the same folder.')

All outputs saved to: C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files
  All quality shots (unclipped, height + biomass) : C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.csv
  All quality points (GeoJSON)                    : C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots.geojson
  All footprints (GeoJSON)                        : C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_footprints.geojson
  Clipped points (GeoJSON)                        : C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_1_gedi_shots_clipped.geojson
  Clipped footprints (GeoJSON)                    : 

## Code for pulling boundary GeoJSON files from ArcGIS Online

In [25]:
from arcgis.gis import GIS

gis = GIS("https://calpoly.maps.arcgis.com", client_id="arcgisPro")

Please sign in to your GIS and paste the code that is obtained below.
If a web browser does not automatically open, please navigate to the URL below yourself instead.
Opening web browser to navigate to: https://calpoly.maps.arcgis.com/sharing/rest/oauth2/authorize?response_type=code&client_id=arcgisPro&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&state=Ew1RF0QlVqMsZ0l6llRqcefE0OmkTd&allow_verification=false


In [29]:
#in ArcGIS online, go to the content page of the feature layer desired to be downloaded onto local machine. 
# in the URL address of that content page, copy the item ID (the long string of letters and numbers after "id=") and paste it below
# do not include #overview
# item = gis.content.get("paste the item id here")
item = gis.content.get("d5d83b32da2f4fc3a974719608a806b4")
flayer = item.layers[0]

# ensure the output spatial reference is WGS84 (EPSG:4326)
fs = flayer.query(where="1=1", out_fields="*", out_sr=4326)
sdf = fs.sdf

# Drop any rows with missing geometry before converting
sdf = sdf[sdf['SHAPE'].notna()].copy()

# Convert each Esri geometry to a shapely geometry, then build a GeoDataFrame
import geopandas as gpd

sdf['geometry'] = sdf['SHAPE'].apply(lambda g: g.as_shapely)
gdf_boundary = gpd.GeoDataFrame(sdf.drop(columns='SHAPE'), geometry='geometry', crs='EPSG:4326')

# paste in the name of the output file desired to be saved on local machine.
# file gets saved to the same folder as this notebook, unless you specify a different path in the filename.
# ex. with open ("name_output_file.geojson", "w") as f:
# or  with open ("C:\\Users\\username\\Documents\\name_output_file.geojson", "w") as f: 
save_path = "C:\\Users\\davisk10\\OneDrive - Cal Poly\\Tree Biomass Estimation Research - Documents\\CODE\\Bartleson_Ranch_Output_Files\\bartleson_crops_boundaries.geojson"
gdf_boundary.to_file(save_path, driver='GeoJSON')

print(f"Saved {len(gdf_boundary)} boundary feature(s) to {save_path}")

Saved 16 boundary feature(s) to C:\Users\davisk10\OneDrive - Cal Poly\Tree Biomass Estimation Research - Documents\CODE\Bartleson_Ranch_Output_Files\bartleson_crops_boundaries.geojson
